# NHS PROMs — Hip Replacement: Data Preparation

**Source data:** `notebooks/Lalita/UnderstandingData/external/`  
**Output (prepared dataset):** `data/interim/hip-provider.parquet`  
**Years covered:** 2016/17 · 2017/18 · 2018/19  

This notebook cleans and prepares the raw NHS PROMs Hip Replacement Provider-level data  
for downstream modelling. Each step documents *what* is done and *why*, grounding decisions  
in the NHS Digital PROMs Technical Specification and the clinical literature.

## 0 · Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

## 1 · Load raw data

Three annual CSV files (2016/17, 2017/18, 2018/19) are concatenated into a single dataframe.  
File names vary slightly across years (singular vs plural); paths are explicit to avoid silent errors.

In [ ]:
# Base directory — external pre-extracted CSVs
EXTERNAL = Path('../../Lalita/UnderstandingData/external')

# Output path for the cleaned, prepared dataset
OUTPUT = Path('../../../data/interim/hip-provider.parquet')

csv_paths = [
    EXTERNAL / 'Hip Replacement Provider 1617.csv',
    EXTERNAL / 'Hip Replacements Provider 1718.csv',   # note: plural in 1718 filename
    EXTERNAL / 'Hip Replacement Provider 1819.csv',
]

# Read and concatenate — low_memory=False prevents dtype inference warnings
# on columns with mixed types (e.g. '*' suppression codes alongside numbers)
frames = [pd.read_csv(p, sep=';', low_memory=False) for p in csv_paths]
df = pd.concat(frames, ignore_index=True)

print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')

## 2 · Initial exploration

In [ ]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
df.head(3)

## 3 · Standardise missing-value codes → `NaN`

The NHS PROMs data uses several conventions for missing / suppressed values:
- **`9`** — the dictionary-defined missing code for most categorical fields
- **`*`** — statistical suppression applied to small counts (< 5) by NHS Digital
- **`999`** — occasionally used for out-of-range / missing numeric entries
- **`''`** — empty string from CSV export

Converting all of these to `NaN` before any imputation or deletion step ensures that
downstream pandas methods (`.isna()`, `.fillna()`, `.dropna()`) behave consistently.

In [ ]:
df.replace({9: np.nan, '*': np.nan, '': np.nan, 999: np.nan}, inplace=True)

total_missing = df.isna().sum().sum()
print(f'Total missing cells after standardisation: {total_missing:,}')

## 4 · Comorbidity flags — treat missing as absent

These 12 columns are binary indicators (1 = condition present, 9 = not reported).  
In line with standard epidemiological practice for administrative datasets, a missing
comorbidity flag is treated as *not present* (0) rather than excluded.  
Rationale: patients with unrecorded comorbidities are unlikely to differ systematically
from patients who answered 'No'; dropping these rows would introduce selection bias.

In [ ]:
COMORBIDITY_COLS = [
    'Arthritis', 'Cancer', 'Circulation', 'Depression', 'Diabetes',
    'Heart Disease', 'High Bp', 'Kidney Disease', 'Liver Disease',
    'Lung Disease', 'Nervous System', 'Stroke'
]

for col in COMORBIDITY_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

print('Comorbidity prevalence (proportion of patients):')
print(df[COMORBIDITY_COLS].mean().round(3))

## 5 · Oxford Hip Score (OHS) pre-op items — drop incomplete responders

The 12 Oxford Hip Score items are the primary disease-specific measure for hip replacement.
Each item is mandatory for the score to be clinically valid.  
Missing rates are very low (< 1 %), so listwise deletion is preferred over imputation:
- Imputing item-level OHS responses would introduce noise into the most predictive feature group.
- The literature (Devlin & Appleby, 2010) requires a complete 12-item response for a valid OHS.

In [ ]:
OHS_PRE_ITEMS = [
    'Hip Replacement Pre-Op Q Pain',
    'Hip Replacement Pre-Op Q Sudden Pain',
    'Hip Replacement Pre-Op Q Night Pain',
    'Hip Replacement Pre-Op Q Washing',
    'Hip Replacement Pre-Op Q Transport',
    'Hip Replacement Pre-Op Q Dressing',
    'Hip Replacement Pre-Op Q Shopping',
    'Hip Replacement Pre-Op Q Walking',
    'Hip Replacement Pre-Op Q Limping',
    'Hip Replacement Pre-Op Q Stairs',
    'Hip Replacement Pre-Op Q Standing',
    'Hip Replacement Pre-Op Q Work',
]

n_before = len(df)
df = df.dropna(subset=OHS_PRE_ITEMS)
print(f'Removed {n_before - len(df):,} rows with any missing OHS pre-op item.')
print(f'Remaining rows: {len(df):,}')

## 6 · Derive OHS pre-op and post-op scores from items

The Oxford Hip Score is the simple sum of its 12 items (range 0–48).  
Where the composite score is missing but all 12 items are present (data entry omission),
it is back-calculated rather than left null — preserving these rows for modelling.

In [ ]:
# Pre-op OHS score: fill from items where the score cell is empty
df['Hip Replacement Pre-Op Q Score'] = df['Hip Replacement Pre-Op Q Score'].fillna(
    df[OHS_PRE_ITEMS].sum(axis=1)
)

# Post-op OHS items (same 12 dimensions, post-operative timepoint)
OHS_POST_ITEMS = [
    'Hip Replacement Post-Op Q Pain',
    'Hip Replacement Post-Op Q Sudden Pain',
    'Hip Replacement Post-Op Q Night Pain',
    'Hip Replacement Post-Op Q Washing',
    'Hip Replacement Post-Op Q Transport',
    'Hip Replacement Post-Op Q Dressing',
    'Hip Replacement Post-Op Q Shopping',
    'Hip Replacement Post-Op Q Walking',
    'Hip Replacement Post-Op Q Limping',
    'Hip Replacement Post-Op Q Stairs',
    'Hip Replacement Post-Op Q Standing',
    'Hip Replacement Post-Op Q Work',
]

# Only fill where all 12 post-op items exist (sum is only valid with a complete response)
post_complete = df[OHS_POST_ITEMS].notna().all(axis=1)
df.loc[post_complete, 'Hip Replacement Post-Op Q Score'] = (
    df.loc[post_complete, 'Hip Replacement Post-Op Q Score']
      .fillna(df.loc[post_complete, OHS_POST_ITEMS].sum(axis=1))
)

print('OHS pre-op score — missing after imputation:',
      df['Hip Replacement Pre-Op Q Score'].isna().sum())
print('OHS post-op score — missing after imputation:',
      df['Hip Replacement Post-Op Q Score'].isna().sum())

## 7 · EQ-5D dimensions — drop rows with missing pre-op or post-op responses

The EQ-5D (5 dimensions × 3 levels) is the generic quality-of-life instrument used
alongside the OHS. Each dimension is required to compute a valid EQ-5D utility index.

**Why listwise deletion and not imputation?**  
Valid EQ-5D levels range from 1 to 3; there is no valid '0'. Imputing a value outside
the allowed range would make the index calculation invalid. Imputing within-range values
risks masking genuine 'non-responders', which is a clinically distinct sub-group.
Missing rates are 2–4 %, so the data loss is acceptable.

In [ ]:
EQ5D_DIMS = [
    'Pre-Op Q Mobility',   'Pre-Op Q Self-Care',  'Pre-Op Q Activity',
    'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety',
    'Post-Op Q Mobility',  'Post-Op Q Self-Care', 'Post-Op Q Activity',
    'Post-Op Q Discomfort','Post-Op Q Anxiety',
]

n_before = len(df)
df = df.dropna(subset=EQ5D_DIMS)
print(f'Removed {n_before - len(df):,} rows with missing EQ-5D dimension responses.')
print(f'Remaining rows: {len(df):,}')

## 8 · Pre-Op Q Assisted — mode imputation (missing → 2 = No)

This flag records whether the patient needed help to complete the questionnaire  
(1 = Yes, 2 = No). Missing rate is ~1 %. The dominant response is 'No', and  
patients who did not record this field are indistinguishable from self-completers,  
making mode imputation safe for this administrative field.

In [ ]:
df['Pre-Op Q Assisted'] = df['Pre-Op Q Assisted'].fillna(2)  # 2 = No
print('Pre-Op Q Assisted — missing after imputation:',
      df['Pre-Op Q Assisted'].isna().sum())

## 9 · Pre-Op Q Symptom Period — listwise deletion

Symptom duration is a clinically meaningful predictor of post-operative outcome:  
longer pre-operative suffering is associated with worse post-op scores.  
Because this variable has strong predictive power, imputation (even mode) risks  
introducing noise into a high-signal feature. Missing rate is < 1 %; the row  
count lost is negligible.

In [ ]:
n_before = len(df)
df = df.dropna(subset=['Pre-Op Q Symptom Period'])
print(f'Removed {n_before - len(df):,} rows with missing Pre-Op Q Symptom Period.')
print(f'Remaining rows: {len(df):,}')

## 10 · Pre-Op Q Previous Surgery — mode imputation (missing → 2 = No)

Whether the patient had previous surgery on the same joint  
(1 = Yes, 2 = No). Missing rate is < 1 %; mode (No) imputation is appropriate  
as most patients are primary procedures.

In [ ]:
df['Pre-Op Q Previous Surgery'] = df['Pre-Op Q Previous Surgery'].fillna(2)  # 2 = No
print('Pre-Op Q Previous Surgery — missing after imputation:',
      df['Pre-Op Q Previous Surgery'].isna().sum())

## 11 · Pre-Op Q Living Arrangements — retain 'Unknown' as a category

Living situation (1 = with family/partner, 2 = alone, 3 = nursing home, 4 = other)  
is a social support indicator. Patients who did not answer this question may represent  
a distinct social group (e.g. reluctance to disclose living alone).  
Retaining the non-response as a separate category (9 = 'Not Specified') preserves  
this potential signal rather than arbitrarily assigning them to a living group.

In [ ]:
# Fill NaN with 9 to represent 'Not Specified' as an explicit category
df['Pre-Op Q Living Arrangements'] = df['Pre-Op Q Living Arrangements'].fillna(9)
print('Living Arrangements value counts:')
print(df['Pre-Op Q Living Arrangements'].value_counts())

## 12 · Pre-Op Q Disability — retain 'Not Disclosed' as a category

Whether the patient has a long-term disability (1 = Yes, 2 = No).  
This has the highest missingness of any non-EQ-5D variable (~6 %).  
Patients may deliberately not self-identify as disabled; that choice is  
potentially informative. Treating non-response as a third level (9 = 'Not Disclosed')  
avoids introducing imputation bias on a high-missingness, sensitive question.

In [ ]:
# Fill NaN with 9 to represent 'Not Disclosed' as an explicit category
df['Pre-Op Q Disability'] = df['Pre-Op Q Disability'].fillna(9)
print('Disability value counts:')
print(df['Pre-Op Q Disability'].value_counts())

## 13 · Derive EQ-5D Index from profile (UK value set)

The EQ-5D-3L utility index is calculated from the five-digit profile using the  
UK time trade-off value set (Dolan et al., 1997).  

This function is used to:
1. **Verify** the recorded index values against the formula (quality check).
2. **Impute** any rows where the index is missing but the five dimension values exist —
   avoiding unnecessary data loss.

In [ ]:
def eq5d_index_from_profile(profile) -> float:
    """Calculate EQ-5D-3L utility index using the UK (Dolan 1997) value set.

    Parameters
    ----------
    profile : int or str
        Five-digit EQ-5D profile, e.g. 11221.  Each digit is 1, 2 or 3.

    Returns
    -------
    float or NaN if the profile is invalid.
    """
    if pd.isna(profile) or len(str(int(profile))) != 5:
        return np.nan
    try:
        digits = [int(d) for d in str(int(profile))]
    except (ValueError, TypeError):
        return np.nan
    if any(d not in (1, 2, 3) for d in digits):
        return np.nan

    # UK TTO value set constants
    CONSTANT = 0.081
    N3_PENALTY = 0.269  # applied if any dimension = 3
    WEIGHTS = [
        {'2': 0.069, '3': 0.314},  # Mobility
        {'2': 0.104, '3': 0.214},  # Self-Care
        {'2': 0.036, '3': 0.094},  # Usual Activities
        {'2': 0.123, '3': 0.386},  # Pain / Discomfort
        {'2': 0.071, '3': 0.236},  # Anxiety / Depression
    ]

    deduction = CONSTANT
    has_level_3 = False
    for i, level in enumerate(digits):
        if level == 2:
            deduction += WEIGHTS[i]['2']
        elif level == 3:
            deduction += WEIGHTS[i]['3']
            has_level_3 = True
    if has_level_3:
        deduction += N3_PENALTY

    return round(1.0 - deduction, 6)

In [ ]:
PRE_EQ5D_DIMS  = ['Pre-Op Q Mobility',  'Pre-Op Q Self-Care',  'Pre-Op Q Activity',
                   'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety']
POST_EQ5D_DIMS = ['Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity',
                   'Post-Op Q Discomfort','Post-Op Q Anxiety']

# Fill missing profiles by concatenating the five dimension scores
df['Pre-Op Q EQ5D Index Profile'] = df['Pre-Op Q EQ5D Index Profile'].fillna(
    df[PRE_EQ5D_DIMS].astype(int).astype(str).agg(''.join, axis=1)
)
df['Post-Op Q EQ5D Index Profile'] = df['Post-Op Q EQ5D Index Profile'].fillna(
    df[POST_EQ5D_DIMS].astype(int).astype(str).agg(''.join, axis=1)
)

# Fill missing index values from the (now complete) profiles
df['Pre-Op Q EQ5D Index'] = df['Pre-Op Q EQ5D Index'].fillna(
    df['Pre-Op Q EQ5D Index Profile'].apply(eq5d_index_from_profile)
)
df['Post-Op Q EQ5D Index'] = df['Post-Op Q EQ5D Index'].fillna(
    df['Post-Op Q EQ5D Index Profile'].apply(eq5d_index_from_profile)
)

print('Pre-Op EQ5D Index  — missing:', df['Pre-Op Q EQ5D Index'].isna().sum())
print('Post-Op EQ5D Index — missing:', df['Post-Op Q EQ5D Index'].isna().sum())

## 14 · Drop post-operative detail columns (keep outcome scores only)

The 12 individual OHS post-op item responses and most post-operative questionnaire  
fields are recorded *after* surgery and are therefore leakage features — they cannot  
be known at the time of prediction. Only the composite outcome scores (OHS and EQ-5D)  
are retained as target variables.

Columns removed:
- 12 OHS post-op item responses (individual Oxford Hip Score dimensions)
- Post-op admin fields (assisted, readmitted, satisfaction, complications, etc.)
- Predicted score columns (model artefacts from NHS Digital's own regression)
- `Procedure` (constant — all rows are Hip Replacement)
- `CSVYear` (redundant with `Year`)

In [ ]:
# Outcome columns to KEEP (target variables)
KEEP_POST_OP = {
    'Post-Op Q EQ5D Index Profile',
    'Post-Op Q EQ5D Index',
    'Post-Op Q EQ VAS',
    'Hip Replacement Post-Op Q Score',
}

# Drop all other post-op columns and NHS-modelled predicted columns
drop_cols = [
    col for col in df.columns
    if ('Post-Op' in col and col not in KEEP_POST_OP)
    or 'Predicted' in col
    or col in {'Procedure', 'CSVYear'}
]

df = df.drop(columns=drop_cols, errors='ignore')
print(f'Dropped {len(drop_cols)} columns. Remaining: {df.shape[1]}')
print('Kept columns:', df.columns.tolist())

## 15 · Age Band and Gender — listwise deletion

Both `Age Band` and `Gender` are suppressed together in the raw data (marked `*`)  
for patient privacy when a cell would contain < 5 observations.  
They account for ~7 % of rows. Because age and gender are key casemix-adjustment  
variables (NHS Digital uses them in all PROMs adjustment models), imputation is  
not appropriate — any imputed demographic would distort case-mix analysis.  
The data loss is acceptable given the overall sample size.

In [ ]:
n_before = len(df)
df = df.dropna(subset=['Age Band', 'Gender'])
print(f'Removed {n_before - len(df):,} rows with suppressed Age Band / Gender.')
print(f'Remaining rows: {len(df):,}')

## 16 · Encode and label categorical variables

Four encoding steps taken directly from the Gino.ipynb exploratory analysis, applied here
in the final preparation pipeline so the saved dataset is ready for modelling without
any further recoding:

1. **Binary recoding** — `Pre-Op Q Assisted` and `Pre-Op Q Previous Surgery` use `2` for "No"
   in the raw dictionary. This is recoded to `0` (standard binary: 0 = No, 1 = Yes) so these
   columns behave like all other 0/1 flags in the dataset.

2. **Gender labels** — Raw codes (`"1"`, `"2"`, `"0"`) are mapped to readable strings
   (`Male`, `Female`, `Not known`) to prevent silent errors in downstream groupby operations.

3. **Living Arrangements labels** — Numeric codes are mapped to the full descriptive strings
   from the NHS PROMs data dictionary, including the `9 = 'Not Specified'` category added in
   Step 11.

4. **Explicit numeric casting** — Columns that should be `float64` but may still be `object`
   dtype (due to mixed-type inference across the three annual CSV files) are forced to numeric.
   `errors='coerce'` converts any residual non-numeric strings to `NaN` rather than raising.

In [ ]:
# ── 16a | Binary recoding ──────────────────────────────────────────────────────
# Raw data dictionary: 1 = Yes, 2 = No.  Recode 2 → 0 so all binary flags
# follow the same convention (0 = No, 1 = Yes) expected by sklearn and statsmodels.

df['Pre-Op Q Assisted']        = df['Pre-Op Q Assisted'].replace(2, 0)
df['Pre-Op Q Previous Surgery'] = df['Pre-Op Q Previous Surgery'].replace(2, 0)

print('Pre-Op Q Assisted value counts (0=No, 1=Yes):')
print(df['Pre-Op Q Assisted'].value_counts())
print('\nPre-Op Q Previous Surgery value counts (0=No, 1=Yes):')
print(df['Pre-Op Q Previous Surgery'].value_counts())

In [ ]:
# ── 16b | Gender labels ────────────────────────────────────────────────────────
# Raw codes arrive as strings ("1", "2", "0") or occasionally integers after
# CSV concatenation.  Both forms are handled to avoid silent NaN introduction.
# Source: NHS PROMs data dictionary — Gender field.

GENDER_MAP = {
    '1': 'Male',   1: 'Male',
    '2': 'Female', 2: 'Female',
    '0': 'Not known', 0: 'Not known',
}
df['Gender'] = df['Gender'].replace(GENDER_MAP)

print('Gender distribution:')
print(df['Gender'].value_counts(normalize=True).round(3))

In [ ]:
# ── 16c | Living Arrangements labels ──────────────────────────────────────────
# Maps numeric codes to full descriptive strings from the NHS PROMs dictionary.
# Code 9 ('Not Specified') was introduced in Step 11 for patients who did not answer.

LIVING_MAP = {
    1: 'Living with partner/spouse/family/friends',
    2: 'Alone',
    3: 'Living in a nursing home, hospital or other long-term care home',
    4: 'Other',
    9: 'Not Specified',
}
df['Pre-Op Q Living Arrangements'] = df['Pre-Op Q Living Arrangements'].replace(LIVING_MAP)

print('Living Arrangements distribution:')
print(df['Pre-Op Q Living Arrangements'].value_counts(normalize=True).round(3))

In [ ]:
# ── 16d | Explicit numeric casting ────────────────────────────────────────────
# After concatenating three annual CSVs, pandas sometimes infers object dtype for
# columns that contain a mix of numeric values and residual string artefacts.
# pd.to_numeric(..., errors='coerce') forces float64 and converts any remaining
# non-numeric strings to NaN (which are then visible in the Step 17 summary).

NUMERIC_COLS = [
    # EQ-5D dimensions (already confirmed numeric, but cast defensively)
    'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity',
    'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety',
    # EQ-5D derived scores
    'Pre-Op Q EQ5D Index', 'Post-Op Q EQ5D Index',
    'Pre-Op Q EQ VAS',     'Post-Op Q EQ VAS',
    # Administrative / casemix fields
    'Pre-Op Q Assisted', 'Pre-Op Q Assisted By',
    'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery',
    'Pre-Op Q Disability', 'Revision Flag',
    # Oxford Hip Score — 12 pre-op items + composite
    'Hip Replacement Pre-Op Q Pain',        'Hip Replacement Pre-Op Q Sudden Pain',
    'Hip Replacement Pre-Op Q Night Pain',  'Hip Replacement Pre-Op Q Washing',
    'Hip Replacement Pre-Op Q Transport',   'Hip Replacement Pre-Op Q Dressing',
    'Hip Replacement Pre-Op Q Shopping',    'Hip Replacement Pre-Op Q Walking',
    'Hip Replacement Pre-Op Q Limping',     'Hip Replacement Pre-Op Q Stairs',
    'Hip Replacement Pre-Op Q Standing',    'Hip Replacement Pre-Op Q Work',
    'Hip Replacement Pre-Op Q Score',
    # Outcome (target variable)
    'Hip Replacement Post-Op Q Score',
]

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Numeric casting complete.')
print('\nDtype check for score columns:')
print(df[[c for c in NUMERIC_COLS if c in df.columns]].dtypes)

## 17 · Final data summary

In [ ]:
print('=== Final dataset ===')
print(f'Rows:    {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'\nRemaining missing values:')
missing = df.isna().sum()
print(missing[missing > 0])
df.describe(include='all')

## 18 · Save prepared dataset

The cleaned dataset is saved as **Parquet** (columnar format) to  
`data/interim/hip-provider.parquet`.

Parquet is preferred over CSV for interim data because:
- dtypes are preserved exactly (no re-parsing needed)
- file size is significantly smaller due to columnar compression
- read/write speeds are faster for downstream notebooks

In [ ]:
# Convert object columns to string for Parquet compatibility
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('string')

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT, index=False)
print(f'Saved prepared dataset to: {OUTPUT.resolve()}')
print(f'File size: {OUTPUT.stat().st_size / 1024:.1f} KB')